# Chapter 21 — Multilinear Algebra

Companion notebook to Chapter 21 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

A $k$-linear form on $\mathbb R^n$ is stored as its array of components, of
shape $(n,\dots,n)$: the entry at $(i_1,\dots,i_k)$ is the value on
$(e_{i_1},\dots,e_{i_k})$. Everything is deterministic except the random
test tensors, which are seeded.

In [1]:
import itertools
import math
import warnings

import torch
from torch.func import jacrev

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def sgn(perm):
    # (-1)^N, N = number of pairs i < j with perm[i] > perm[j]
    n_inv = sum(1 for i in range(len(perm)) for j in range(i + 1, len(perm)) if perm[i] > perm[j])
    return -1 if n_inv % 2 else 1


def act(perm, t):
    # (sigma T)(v_1, ..., v_k) = T(v_sigma(1), ..., v_sigma(k)):  components (sigma T)[i] = T[i_sigma(1), ...]
    inv = [0] * len(perm)
    for a, b in enumerate(perm):
        inv[b] = a
    return t.permute(*inv)


def antisym(t):
    # A(T) = sum over all permutations of sgn(sigma) sigma T
    k = t.dim()
    return sum(sgn(p) * act(p, t) for p in itertools.permutations(range(k)))


def tensor_product(s, t):
    return torch.tensordot(s, t, dims=0)


def wedge(w, e):
    k, l = w.dim(), e.dim()
    return antisym(tensor_product(w, e)) / (math.factorial(k) * math.factorial(l))


def evaluate(t, *vectors):
    for v in vectors:
        t = torch.tensordot(v, t, dims=([0], [0]))
    return t


def random_form(n, k):
    return antisym(torch.randn(*([n] * k))) / math.factorial(k)

2.15.0+cu132


## 1. Tensors, components and change of basis (Theorem 21.4, Table 21.1, Exercise 21.1)

The same $2\times2$ array, read as an operator and as a bilinear form, in
the basis $e'_1 = e_1$, $e'_2 = e_1 + e_2$.

In [2]:
arr = T([[2.0, 1.0], [0.0, 1.0]])
P = T([[1.0, 1.0], [0.0, 1.0]])              # columns: the new basis vectors
Pinv = torch.linalg.inv(P)

operator_new = Pinv @ arr @ P
form_new = P.T @ arr @ P
print("operator in e':\n", operator_new)
print("bilinear form in e':\n", form_new)
assert torch.allclose(operator_new, T([[2.0, 2.0], [0.0, 1.0]]))
assert torch.allclose(form_new, T([[2.0, 3.0], [2.0, 4.0]]))
print(f"operator: trace {arr.trace():.0f} -> {operator_new.trace():.0f}, det {torch.det(arr):.0f} -> {torch.det(operator_new):.0f}")
print(f"form:     trace {arr.trace():.0f} -> {form_new.trace():.0f}, det {torch.det(arr):.0f} -> {torch.det(form_new):.0f}"
      f"  (det P = {torch.det(P):.0f})")

operator in e':
 tensor([[2., 2.],
        [0., 1.]])
bilinear form in e':
 tensor([[2., 3.],
        [2., 4.]])
operator: trace 3 -> 3, det 2 -> 2
form:     trace 3 -> 6, det 2 -> 2  (det P = 1)


The general rule: one factor $P^{-1}$ for each covector slot and one factor
$P$ for each vector slot. A random tensor of type $(1, 2)$ on $\mathbb R^4$
is evaluated on the same vector and covectors in two bases; the number does
not change. The two contractions over a covector slot and a vector slot are
invariant; the "contraction" over the two vector slots is not.

In [3]:
n = 4
Tn = torch.randn(n, n, n)                      # type (1, 2): one covector slot, then two vector slots
P = torch.randn(n, n) + 2 * torch.eye(n)
Pinv = torch.linalg.inv(P)
T_new = torch.einsum("ia,bj,ck,abc->ijk", Pinv, P, P, Tn)

phi, u, v = torch.randn(n), torch.randn(n), torch.randn(n)      # components in the basis e
phi_new, u_new, v_new = phi @ P, Pinv @ u, Pinv @ v              # covector: row times P; vectors: P^-1 times column
value = torch.einsum("abc,a,b,c->", Tn, phi, u, v)
value_new = torch.einsum("abc,a,b,c->", T_new, phi_new, u_new, v_new)
print(f"T(phi, u, v) = {value:.10f} in e,  {value_new:.10f} in e'")
assert torch.allclose(value, value_new)

good = torch.einsum("iij->j", Tn)              # contraction covector slot / first vector slot: a covector
good_new = torch.einsum("iij->j", T_new)
assert torch.allclose(good_new, good @ P)      # it transforms as a covector should
bad = torch.einsum("ijj->i", Tn)               # summing over two vector slots
bad_new = torch.einsum("ijj->i", T_new)
print("contraction over two slots of the same kind transforms as a vector?",
      torch.allclose(bad_new, Pinv @ bad))
assert not torch.allclose(bad_new, Pinv @ bad)

A, B = torch.randn(n, n), torch.randn(n, n)
assert torch.allclose(torch.einsum("ij,jk->ik", A, B), A @ B)   # AB is a contraction of A (x) B
assert torch.allclose(torch.einsum("ii->", Pinv @ A @ P), A.trace())

T(phi, u, v) = -0.2767692455 in e,  -0.2767692455 in e'
contraction over two slots of the same kind transforms as a vector? False


**Which numbers mean something** (first machine-learning box of §21.1). A
loss with a minimum at the origin, in two parametrizations that differ by
the scaling $\theta = P\theta'$. The Hessian array changes by
$P^\top HP$: its eigenvalues change, their signs do not. With a metric $G$,
transformed in the same way, the eigenvalues of $G^{-1}H$ are the same in
both.

In [4]:
M = torch.randn(n, n)
H = M @ M.T + 0.1 * torch.eye(n)                # a positive definite Hessian
loss = lambda th: 0.5 * th @ H @ th
P = torch.diag(T([10.0, 1.0, 0.1, 1.0]))        # rescale two parameters
loss_new = lambda th_new: loss(P @ th_new)

H_auto = jacrev(jacrev(loss_new))(torch.zeros(n))
assert torch.allclose(H_auto, P.T @ H @ P)
eig, eig_new = torch.linalg.eigvalsh(H), torch.linalg.eigvalsh(H_auto)
print("eigenvalues of the Hessian array:      ", [f"{x:.3f}" for x in eig.tolist()])
print("after reparametrization:               ", [f"{x:.3f}" for x in eig_new.tolist()])
assert not torch.allclose(eig, eig_new) and (eig > 0).all() and (eig_new > 0).all()

Gm = torch.randn(n, n)
G = Gm @ Gm.T + torch.eye(n)                    # a metric, in the first parametrization
G_new = P.T @ G @ P
rel = torch.linalg.eigvals(torch.linalg.solve(G, H)).real.sort().values
rel_new = torch.linalg.eigvals(torch.linalg.solve(G_new, H_auto)).real.sort().values
print("eigenvalues of G^-1 H:                 ", [f"{x:.4f}" for x in rel.tolist()])
print("after reparametrization:               ", [f"{x:.4f}" for x in rel_new.tolist()])
assert torch.allclose(rel, rel_new)

eigenvalues of the Hessian array:       ['0.134', '0.960', '3.710', '5.750']
after reparametrization:                ['0.016', '0.164', '2.717', '388.316']
eigenvalues of G^-1 H:                  ['0.0535', '0.3444', '0.9730', '2.6457']
after reparametrization:                ['0.0535', '0.3444', '0.9730', '2.6457']


The rescaling symmetry of a ReLU network: multiplying the first layer by
$\alpha$ and the second by $1/\alpha$ leaves the function unchanged and
changes the largest eigenvalue of the Hessian of the loss.

In [5]:
X = torch.randn(64, 3)
y = torch.randn(64)
W1, w2 = torch.randn(5, 3), torch.randn(5)


def net_loss(params):
    A1, a2 = params[:15].reshape(5, 3), params[15:]
    return ((torch.relu(X @ A1.T) @ a2 - y)**2).mean()


for alpha in (1.0, 3.0, 10.0):
    params = torch.cat([(alpha * W1).flatten(), w2 / alpha])
    Hn = jacrev(jacrev(net_loss))(params)
    print(f"alpha = {alpha:4.1f}:  loss = {net_loss(params):.10f}   largest Hessian eigenvalue = "
          f"{torch.linalg.eigvalsh(Hn).max():.4f}")
base = net_loss(torch.cat([W1.flatten(), w2]))
assert abs(net_loss(torch.cat([(10 * W1).flatten(), w2 / 10])) - base) < 1e-12
lam = [torch.linalg.eigvalsh(jacrev(jacrev(net_loss))(torch.cat([(a * W1).flatten(), w2 / a]))).max().item()
       for a in (1.0, 10.0)]
assert lam[1] > 5 * lam[0]

alpha =  1.0:  loss = 19.5988152694   largest Hessian eigenvalue = 16.2489
alpha =  3.0:  loss = 19.5988152694   largest Hessian eigenvalue = 80.7726
alpha = 10.0:  loss = 19.5988152694   largest Hessian eigenvalue = 884.1022


## 2. Permutations and alternating forms (Lemma 21.7, Proposition 21.9, Exercises 21.2 and 21.3)

In [6]:
# sign is multiplicative, on all pairs of permutations of 4 elements
perms = list(itertools.permutations(range(4)))
compose = lambda s, t: tuple(s[t[i]] for i in range(4))          # (s t)(i) = s(t(i))
assert all(sgn(compose(s, t)) == sgn(s) * sgn(t) for s in perms for t in perms)
assert sum(sgn(p) for p in perms) == 0 and sgn((1, 0, 2, 3)) == -1 and sgn((3, 1, 2, 0)) == -1
print("sgn(st) = sgn(s) sgn(t) for the 576 pairs;  sgn(2,3,1) =", sgn((1, 2, 0)), "  sgn(2,1,4,3) =", sgn((1, 0, 3, 2)))
assert sgn((1, 2, 0)) == 1 and sgn((1, 0, 3, 2)) == 1

# the action is compatible with composition: tau(sigma T) = (tau sigma) T
t3 = torch.randn(3, 3, 3)
for s in itertools.permutations(range(3)):
    for t in itertools.permutations(range(3)):
        ts = tuple(t[s[i]] for i in range(3))
        assert torch.allclose(act(t, act(s, t3)), act(ts, t3))

# an alternating form vanishes on dependent vectors
w = random_form(5, 3)
a, b = torch.randn(5), torch.randn(5)
print(f"omega(a, b, 2a - 3b) = {evaluate(w, a, b, 2 * a - 3 * b):.2e}")
assert abs(evaluate(w, a, b, 2 * a - 3 * b)) < 1e-12
assert torch.allclose(evaluate(w, a, b, torch.ones(5)), -evaluate(w, b, a, torch.ones(5)))
assert torch.allclose(antisym(w), 6 * w)                          # A(omega) = k! omega

# Exercise 21.2: symmetric plus alternating
b_arr = T([[2.0, 1.0], [0.0, 1.0]])
sym, alt = (b_arr + b_arr.T) / 2, (b_arr - b_arr.T) / 2
e1, e2 = torch.eye(2)
assert torch.allclose(alt, 0.5 * wedge(e1, e2)) and torch.allclose(sym + alt, b_arr)
print("alternating part of b = 1/2 e1* ^ e2*")

sgn(st) = sgn(s) sgn(t) for the 576 pairs;  sgn(2,3,1) = 1   sgn(2,1,4,3) = 1
omega(a, b, 2a - 3b) = 1.78e-15
alternating part of b = 1/2 e1* ^ e2*


## 3. The wedge product (Definition 21.10, Theorems 21.11 and 21.12, Figures 21.1 and 21.2)

Associativity, graded commutativity and the general formula, on random
forms of $\mathbb R^5$; then the determinant formula for covectors.

In [7]:
n = 5
w1, w2, w3 = random_form(n, 1), random_form(n, 2), random_form(n, 2)
left, right = wedge(wedge(w1, w2), w3), wedge(w1, wedge(w2, w3))
assert torch.allclose(left, right)
general = antisym(tensor_product(tensor_product(w1, w2), w3)) / (1 * 2 * 2)
assert torch.allclose(left, general)
print("(w1 ^ w2) ^ w3 = w1 ^ (w2 ^ w3) = A(w1 (x) w2 (x) w3) / (k1! k2! k3!)")

for k, l in [(1, 1), (1, 2), (2, 2), (2, 3), (1, 3)]:
    a_, b_ = random_form(n, k), random_form(n, l)
    assert torch.allclose(wedge(b_, a_), (-1)**(k * l) * wedge(a_, b_))
print("eta ^ omega = (-1)^(kl) omega ^ eta for (k, l) in (1,1), (1,2), (2,2), (2,3), (1,3)")

# covectors: (phi_1 ^ ... ^ phi_k)(v_1, ..., v_k) = det[phi_i(v_j)]
for k in (2, 3, 4):
    phis = torch.randn(k, n)
    vs = torch.randn(k, n)
    form = phis[0]
    for i in range(1, k):
        form = wedge(form, phis[i])
    lhs = evaluate(form, *vs)
    rhs = torch.det(phis @ vs.T)
    leibniz = sum(sgn(p) * math.prod((phis[i] @ vs[p[i]]).item() for i in range(k))
                  for p in itertools.permutations(range(k)))
    print(f"k = {k}:  wedge = {lhs:+.8f}   det = {rhs:+.8f}   Leibniz = {leibniz:+.8f}")
    assert torch.allclose(lhs, rhs) and abs(leibniz - rhs.item()) < 1e-10

(w1 ^ w2) ^ w3 = w1 ^ (w2 ^ w3) = A(w1 (x) w2 (x) w3) / (k1! k2! k3!)
eta ^ omega = (-1)^(kl) omega ^ eta for (k, l) in (1,1), (1,2), (2,2), (2,3), (1,3)
k = 2:  wedge = +4.04976485   det = +4.04976485   Leibniz = +4.04976485
k = 3:  wedge = +16.69636384   det = +16.69636384   Leibniz = +16.69636384
k = 4:  wedge = -1.56060850   det = -1.56060850   Leibniz = -1.56060850


In [8]:
# Figure 21.1: phi(x, y) = x, psi(x, y) = y - x/2, u = (2, 1), v = (0.5, 2)
phi, psi = T([1.0, 0.0]), T([-0.5, 1.0])
u, v = T([2.0, 1.0]), T([0.5, 2.0])
cells = evaluate(wedge(phi, psi), u, v)
print(f"phi(u) = {phi @ u}, phi(v) = {phi @ v}, psi(u) = {psi @ u}, psi(v) = {psi @ v};  (phi ^ psi)(u, v) = {cells}")
assert cells == 3.5 and evaluate(wedge(phi, psi), v, u) == -3.5

# Figure 21.2 and Example 21.14: e1* ^ e2* in R^3 is the area of the shadow
e = torch.eye(3)
u3, v3 = T([2.2, 0.4, 0.5]), T([0.5, 2.0, 0.9])
shadow = evaluate(wedge(e[0], e[1]), u3, v3)
assert torch.allclose(shadow, u3[0] * v3[1] - u3[1] * v3[0])
print(f"(e1* ^ e2*)(u, v) = {shadow:.2f} = u1 v2 - u2 v1")

phi(u) = 2.0, phi(v) = 0.5, psi(u) = 0.0, psi(v) = 1.75;  (phi ^ psi)(u, v) = 3.5
(e1* ^ e2*)(u, v) = 4.20 = u1 v2 - u2 v1


The basis of $\Lambda^kV^*$: the forms $e_I^*$ are independent, there are
$\binom nk$ of them, and $\omega = \sum_I\omega(e_I)\,e_I^*$. The dimension
is also the rank of the antisymmetrization map on all $k$-linear forms.

In [9]:
def basis_form(n, I):
    e = torch.eye(n)
    f = e[I[0]]
    for i in I[1:]:
        f = wedge(f, e[i])
    return f


n = 5
for k in (1, 2, 3, 4, 5):
    subsets = list(itertools.combinations(range(n), k))
    basis = torch.stack([basis_form(n, I).flatten() for I in subsets])
    assert torch.linalg.matrix_rank(basis) == math.comb(n, k) == len(subsets)
    # rank of T -> A(T) on the n^k-dimensional space of k-linear forms
    if k <= 3:
        images = torch.stack([antisym(t.reshape([n] * k)).flatten() for t in torch.eye(n**k)])
        assert torch.linalg.matrix_rank(images) == math.comb(n, k)
    w = random_form(n, k)
    rebuilt = sum(w[I] * basis_form(n, I) for I in subsets)        # omega(e_I) is the component w[I]
    assert torch.allclose(rebuilt, w)
    print(f"k = {k}:  dim = {math.comb(n, k):2d}   omega = sum of omega(e_I) e_I*  checked")

k = 1:  dim =  5   omega = sum of omega(e_I) e_I*  checked
k = 2:  dim = 10   omega = sum of omega(e_I) e_I*  checked
k = 3:  dim = 10   omega = sum of omega(e_I) e_I*  checked
k = 4:  dim =  5   omega = sum of omega(e_I) e_I*  checked
k = 5:  dim =  1   omega = sum of omega(e_I) e_I*  checked


Corollary 21.13 and Exercises 21.3 to 21.5.

In [10]:
e = torch.eye(4)
e123 = basis_form(4, (0, 1, 2))
assert torch.allclose(wedge(wedge(e[2], e[0]), e[1]), e123)                       # e3* ^ e1* ^ e2*
assert torch.allclose(wedge(wedge(wedge(e[1], e[0]), e[3]), e[2]), basis_form(4, (0, 1, 2, 3)))
print("e3*^e1*^e2* = e1*^e2*^e3*   and   e2*^e1*^e4*^e3* = e1*^e2*^e3*^e4*")

# Exercise 21.4: omega = e1*^e2* + e3*^e4* is not decomposable
omega = wedge(e[0], e[1]) + wedge(e[2], e[3])
sq = wedge(omega, omega)
assert torch.allclose(sq, 2 * basis_form(4, (0, 1, 2, 3)))
phi, psi = torch.randn(4), torch.randn(4)
assert torch.allclose(wedge(wedge(phi, psi), wedge(phi, psi)), torch.zeros(4, 4, 4, 4))
print("omega ^ omega = 2 e1*^e2*^e3*^e4*, while (phi ^ psi) ^ (phi ^ psi) = 0")

# in R^3 every 2-form is decomposable
e = torch.eye(3)
a_, b_, c_ = 0.7, -1.3, 2.1
w = a_ * wedge(e[1], e[2]) + b_ * wedge(e[2], e[0]) + c_ * wedge(e[0], e[1])
assert torch.allclose(wedge(c_ * e[0] - a_ * e[2], e[1] - b_ / c_ * e[2]), w)

# Exercise 21.5
p1, p2, p3 = T([1.0, 2.0, 0.0]), T([0.0, 1.0, -1.0]), T([1.0, 1.0, 1.0])
val = evaluate(wedge(p1, p2), T([1.0, 0.0, 1.0]), T([0.0, 1.0, 1.0]))
triple = wedge(wedge(p1, p2), p3)
print(f"(phi1 ^ phi2)(u, v) = {val:.0f};   phi1 ^ phi2 ^ phi3 = {triple[0, 1, 2]:.0f} e1*^e2*^e3*;  phi3 = phi1 - phi2:",
      torch.allclose(p3, p1 - p2))
assert val == 2 and torch.allclose(triple, torch.zeros(3, 3, 3)) and torch.allclose(p3, p1 - p2)
indep = wedge(wedge(p1, p2), T([1.0, 0.0, 0.0]))
assert indep[0, 1, 2] != 0                      # an independent triple has a non-zero wedge product

e3*^e1*^e2* = e1*^e2*^e3*   and   e2*^e1*^e4*^e3* = e1*^e2*^e3*^e4*
omega ^ omega = 2 e1*^e2*^e3*^e4*, while (phi ^ psi) ^ (phi ^ psi) = 0
(phi1 ^ phi2)(u, v) = 2;   phi1 ^ phi2 ^ phi3 = 0 e1*^e2*^e3*;  phi3 = phi1 - phi2: True


## 4. Pull-back and the determinant (Definitions 21.15 and 21.16, Theorem 21.17, Proposition 21.18, Exercises 21.6 and 21.7)

The pull-back of a $k$-form by a matrix $A$ applies $A$ to each argument.
On $n$-forms of $\mathbb R^n$ it is the multiplication by $\det A$.

In [11]:
def pullback(A, w):
    # (A* w)(v_1, ..., v_k) = w(A v_1, ..., A v_k)
    for axis in range(w.dim()):
        w = torch.tensordot(w, A, dims=([0], [0]))      # contract the leading slot with A, result goes last
    return w


n = 4
A, B = torch.randn(n, n), torch.randn(n, n)
top = basis_form(n, (0, 1, 2, 3))
assert torch.allclose(pullback(A, top), torch.det(A) * top)
assert torch.allclose(pullback(A @ B, top), pullback(B, pullback(A, top)))        # (AB)* = B* A*
print(f"A* (e1*^...^e4*) = det(A) e1*^...^e4*,  det A = {torch.det(A):.6f}")
print(f"det(AB) = {torch.det(A @ B):.6f} = det A det B = {torch.det(A) * torch.det(B):.6f}")
assert torch.allclose(torch.det(A @ B), torch.det(A) * torch.det(B))

w2, w1 = random_form(n, 2), random_form(n, 1)
assert torch.allclose(pullback(A, wedge(w2, w1)), wedge(pullback(A, w2), pullback(A, w1)))
assert torch.allclose(pullback(A, w1), w1 @ A)                                    # k = 1: the dual operator
P = torch.randn(n, n) + 2 * torch.eye(n)
assert torch.allclose(torch.det(torch.linalg.inv(P) @ A @ P), torch.det(A))
assert torch.allclose(torch.det(A.T), torch.det(A))
G = B @ B.T + torch.eye(n)
assert torch.allclose(torch.det(P.T @ G @ P), torch.det(P)**2 * torch.det(G))
print("A*(w ^ eta) = A*w ^ A*eta;  det(P^-1 A P) = det A;  det(P^T G P) = (det P)^2 det G")

singular = A.clone()
singular[:, 3] = singular[:, 0] - 2 * singular[:, 1]
assert torch.allclose(pullback(singular, top), torch.zeros_like(top), atol=1e-12)

A* (e1*^...^e4*) = det(A) e1*^...^e4*,  det A = -0.518212
det(AB) = 0.650086 = det A det B = 0.650086
A*(w ^ eta) = A*w ^ A*eta;  det(P^-1 A P) = det A;  det(P^T G P) = (det P)^2 det G


In [12]:
# Minors and Cauchy-Binet
A = torch.randn(5, 3)                                   # a map from R^3 to R^5
for J in itertools.combinations(range(5), 2):
    pulled = pullback(A, basis_form(5, J))              # a 2-form on R^3
    for I in itertools.combinations(range(3), 2):
        assert torch.allclose(pulled[I], torch.det(A[list(J)][:, list(I)]))
print("the components of A* f_J* are the minors det A[J, I]")

M = torch.randn(6, 3)
minors = [torch.det(M[list(K)]) for K in itertools.combinations(range(6), 3)]
print(f"det(M^T M) = {torch.det(M.T @ M):.8f}   sum of the {len(minors)} squared minors = {sum(m**2 for m in minors):.8f}")
assert torch.allclose(torch.det(M.T @ M), sum(m**2 for m in minors))

# Exercise 21.7
A7 = T([[1.0, 0.0], [1.0, 1.0], [0.0, 2.0]])
m7 = {J: pullback(A7, basis_form(3, J))[0, 1].item() for J in itertools.combinations(range(3), 2)}
print("pull-backs of the basis 2-forms:", {tuple(j + 1 for j in J): v for J, v in m7.items()},
      "  det(M^T M) =", torch.det(A7.T @ A7).item())
assert m7 == {(0, 1): 1.0, (0, 2): 2.0, (1, 2): 2.0} and abs(torch.det(A7.T @ A7) - 9) < 1e-12

# Exercise 21.6: triangular determinant, and the trace as the derivative of the determinant
U = torch.triu(torch.randn(5, 5))
assert torch.allclose(torch.det(U), torch.diagonal(U).prod())
A = torch.randn(5, 5)
d = jacrev(lambda t: torch.det(torch.eye(5) + t * A))(T(0.0))
print(f"d/dt det(I + tA) at 0 = {d:.8f}   tr A = {A.trace():.8f}")
assert torch.allclose(d, A.trace())
Ainv = A + 3 * torch.eye(5)
grad_det = jacrev(torch.det)(Ainv)                      # array of partial derivatives
H = torch.randn(5, 5)
assert torch.allclose((grad_det * H).sum(), torch.det(Ainv) * torch.trace(torch.linalg.solve(Ainv, H)))
S = Ainv @ Ainv.T
assert torch.allclose(jacrev(torch.logdet)(S), torch.linalg.inv(S).T)
print("d det(A) H = det A tr(A^-1 H);   the array of partial derivatives of ln det S is S^-T")

the components of A* f_J* are the minors det A[J, I]
det(M^T M) = 280.77711972   sum of the 20 squared minors = 280.77711972
pull-backs of the basis 2-forms: {(1, 2): 1.0, (1, 3): 2.0, (2, 3): 2.0}   det(M^T M) = 9.0
d/dt det(I + tA) at 0 = -3.46760382   tr A = -3.46760382


d det(A) H = det A tr(A^-1 H);   the array of partial derivatives of ln det S is S^-T


## 5. Orientation (Definition 21.19, Figure 21.3, Exercise 21.8)

In [13]:
e1, e2, e3 = torch.eye(3)
bases = {"(e2, e3, e1)": [e2, e3, e1], "(e2, e1, e3)": [e2, e1, e3], "(e1+e2, e2, -e3)": [e1 + e2, e2, -e3],
         "(e1, e2, -e3)": [e1, e2, -e3]}
signs = {name: torch.det(torch.stack(vecs, dim=1)).item() for name, vecs in bases.items()}
print(signs)
assert [round(s) for s in signs.values()] == [1, -1, -1, -1]

for n in (2, 3, 4, 5):
    assert torch.allclose(torch.det(-torch.eye(n)), T((-1.0)**n))
    nrm = torch.randn(n)
    reflection = torch.eye(n) - 2 * torch.outer(nrm, nrm) / (nrm @ nrm)
    assert torch.allclose(torch.det(reflection), T(-1.0))
print("det(-I) = (-1)^n;  a reflection has determinant -1")

# n-forms change by det P (Theorem 21.17 (4))
P = torch.randn(4, 4)
top = random_form(4, 4)
assert torch.allclose(evaluate(top, *P.T), torch.det(P) * top[0, 1, 2, 3])

{'(e2, e3, e1)': 1.0, '(e2, e1, e3)': -1.0, '(e1+e2, e2, -e3)': -1.0, '(e1, e2, -e3)': -1.0}
det(-I) = (-1)^n;  a reflection has determinant -1


## 6. What an inner product adds (Theorems 21.20 and 21.21, Example 21.22, Exercise 21.9)

The volume form in a non-orthonormal positive basis is
$\sqrt{\det G}\,e_1^*\wedge\dots\wedge e_n^*$: it gives $1$ on a positive
orthonormal basis.

In [14]:
n = 4
Bm = torch.randn(n, n)
G = Bm @ Bm.T + torch.eye(n)                              # Gram matrix of an inner product in the basis e
vol = torch.sqrt(torch.det(G)) * basis_form(n, tuple(range(n)))

L = torch.linalg.cholesky(G)                              # G = L L^T
O = torch.linalg.inv(L).T                                 # columns: an orthonormal basis, in components
assert torch.allclose(O.T @ G @ O, torch.eye(n))
if torch.det(O) < 0:
    O[:, 0] = -O[:, 0]
print(f"vol on a positive orthonormal basis = {evaluate(vol, *O.T):.12f}")
assert torch.allclose(evaluate(vol, *O.T), T(1.0))
Q, _ = torch.linalg.qr(torch.randn(n, n))                 # another orthonormal basis: rotate the first
if torch.det(Q) < 0:
    Q[:, 0] = -Q[:, 0]
assert torch.allclose(evaluate(vol, *(O @ Q).T), T(1.0))

vol on a positive orthonormal basis = 1.000000000000


The Hodge star on $\mathbb R^3$ and on $\mathbb R^4$ with the usual inner
product, built from its defining property
$\alpha\wedge\star\beta = \langle\alpha, \beta\rangle\,\mathrm{vol}$ on
basis forms, then checked on random forms; $\star\star = (-1)^{k(n-k)}$;
and the cross product.

In [15]:
def inner(a, b):
    # inner product of k-forms for the usual inner product: the basis forms e_I* are orthonormal
    return (a * b).sum() / math.factorial(a.dim()) if a.dim() else a * b


def hodge(w, n):
    k = w.dim()
    result = torch.zeros([n] * (n - k)) if n > k else torch.zeros(())
    for I in itertools.combinations(range(n), k):
        Ic = tuple(i for i in range(n) if i not in I)
        coeff = w[I] if k else w
        result = result + coeff * sgn(I + Ic) * (basis_form(n, Ic) if Ic else torch.ones(()))
    return result


for n in (3, 4):
    volume = basis_form(n, tuple(range(n)))
    for k in range(0, n + 1):
        a_ = random_form(n, k) if k else torch.randn(())
        b_ = random_form(n, k) if k else torch.randn(())
        sb = hodge(b_, n)
        lhs = wedge(a_, sb) if (k and k < n) else (a_ * sb if not k else a_ * sb)
        assert torch.allclose(lhs, inner(a_, b_) * volume)
        back = hodge(sb, n) if k < n else hodge(sb.reshape(()), n)
        assert torch.allclose(back, (-1)**(k * (n - k)) * b_)
    print(f"n = {n}:  alpha ^ *beta = <alpha, beta> vol  and  ** = (-1)^(k(n-k))  for k = 0..{n}")

e = torch.eye(3)
assert torch.allclose(hodge(e[0], 3), wedge(e[1], e[2])) and torch.allclose(hodge(e[1], 3), wedge(e[2], e[0]))
assert torch.allclose(hodge(wedge(e[0], e[1]), 3), e[2]) and torch.allclose(hodge(wedge(e[0], e[2]), 3), -e[1])

u, v, w = torch.randn(3), torch.randn(3), torch.randn(3)
cross = hodge(wedge(u, v), 3)                             # the Riesz map is the identity on components here
print("*(u ^ v)       =", [f"{x:+.5f}" for x in cross.tolist()])
print("torch.cross    =", [f"{x:+.5f}" for x in torch.linalg.cross(u, v).tolist()])
assert torch.allclose(cross, torch.linalg.cross(u, v))
assert torch.allclose(cross @ w, torch.det(torch.stack([u, v, w], dim=1)))

n = 3:  alpha ^ *beta = <alpha, beta> vol  and  ** = (-1)^(k(n-k))  for k = 0..3


n = 4:  alpha ^ *beta = <alpha, beta> vol  and  ** = (-1)^(k(n-k))  for k = 0..4
*(u ^ v)       = ['+2.53959', '-0.55019', '+0.70146']
torch.cross    = ['+2.53959', '-0.55019', '+0.70146']


In [16]:
# Exercise 21.9 (2): Gram matrix diag(1, 4) on R^2, standard orientation.
G = torch.diag(T([1.0, 4.0]))
e = torch.eye(2)
vol = torch.sqrt(torch.det(G)) * wedge(e[0], e[1])
o1s, o2s = e[0], 2 * e[1]                                 # dual basis of the orthonormal basis (e1, e2/2)
assert torch.allclose(vol, wedge(o1s, o2s))
Ginv = torch.linalg.inv(G)                                # inner product of covectors
star_e1, star_e2 = 2 * e[1], -0.5 * e[0]
for beta, star_beta in [(e[0], star_e1), (e[1], star_e2)]:
    for alpha in (e[0], e[1]):
        assert torch.allclose(wedge(alpha, star_beta), (alpha @ Ginv @ beta) * vol)
print("vol = 2 e1*^e2*,  *e1* = 2 e2*,  *e2* = -1/2 e1*")

vol = 2 e1*^e2*,  *e1* = 2 e2*,  *e2* = -1/2 e1*


## 7. Determinants as volumes (second machine-learning box of §21.6)

For feature vectors in $\mathbb R^n$, the Gram determinant of a subset is
the squared volume it spans; for two unit vectors it is $\sin^2$ of their
angle. Near-duplicates give a small determinant: a
determinantal point process, which makes the probability of a subset
proportional to this determinant, prefers diverse subsets. With five items
of which two are almost equal, the probabilities of all subsets of size two
are computed; they add up to $1$ by the Cauchy–Binet formula.

In [17]:
feats = torch.randn(5, 3)
feats[1] = feats[0] + 0.05 * torch.randn(3)               # item 2 is almost item 1
feats = feats / feats.norm(dim=1, keepdim=True)           # unit vectors: only the angles matter
Lmat = feats @ feats.T                                    # Gram matrix of the items
pairs = list(itertools.combinations(range(5), 2))
weights = T([torch.det(Lmat[list(S)][:, list(S)]).item() for S in pairs])
probs = weights / weights.sum()
for S, p in zip(pairs, probs.tolist()):
    print(f"items {tuple(i + 1 for i in S)}: probability {p:.4f}")
assert probs.argmin() == 0                                # the near-duplicate pair is the least likely
# the squared area by the Gram determinant and by the cross product
i, j = 2, 4
assert torch.allclose(weights[pairs.index((i, j))], torch.linalg.cross(feats[i], feats[j]).pow(2).sum())
# normalization: the sum of all 2 x 2 principal minors is the second elementary symmetric function
ev = torch.linalg.eigvalsh(Lmat)
e2 = sum(ev[a] * ev[b] for a, b in itertools.combinations(range(5), 2))
assert torch.allclose(weights.sum(), e2)
print(f"sum of the weights = {weights.sum():.6f} = second elementary symmetric function of the eigenvalues")

items (1, 2): probability 0.0001
items (1, 3): probability 0.0312
items (1, 4): probability 0.0021
items (1, 5): probability 0.2233
items (2, 3): probability 0.0302
items (2, 4): probability 0.0033
items (2, 5): probability 0.2226
items (3, 4): probability 0.0404
items (3, 5): probability 0.2230
items (4, 5): probability 0.2236
sum of the weights = 4.466029 = second elementary symmetric function of the eigenvalues


## Summary

* One $P^{-1}$ per covector slot and one $P$ per vector slot; contracting a
  covector slot with a vector slot is invariant, summing over two slots of
  the same kind is not.
* The eigenvalues of a Hessian array change under reparametrization, and
  under the rescaling symmetry of a ReLU network; those of $G^{-1}H$ do not.
* The wedge product is associative and graded commutative, and on covectors
  it is a determinant; $\dim\Lambda^k = \binom nk$.
* Pull-back of the top form is multiplication by the determinant; minors are
  components of pull-backs; $\det(M^\top M)$ is the sum of the squared
  minors; the trace is the derivative of the determinant.
* $\mathrm{vol} = \sqrt{\det G}\,e_1^*\wedge\dots\wedge e_n^*$; the Hodge star
  satisfies $\alpha\wedge\star\beta = \langle\alpha,\beta\rangle\mathrm{vol}$
  and gives the cross product in $\mathbb R^3$.